# 🐼 Módulo 01: Pandas

Qué es pandas, Series, operaciones vectorizadas y tu primer contacto con tablas reales.

**PandasQuest** · versión web interactiva: [https://rosalesluciano.github.io/pandas-quest/](https://rosalesluciano.github.io/pandas-quest/#/m/m01)

**Cómo funciona:**
1. Ejecuta la celda ⚙️ de abajo (carga las tablas y el corrector).
2. En cada ejercicio escribe tu código y guarda la respuesta en `resultado`.
3. Ejecuta la celda: `comprobar()` te dirá si está bien. 🎯

¿Atascado? `ver_solucion("id")` muestra la solución (¡pero inténtalo antes! 😉)

In [ ]:
#@title ⚙️ 1) Ejecuta esta celda primero (carga datos y corrector) { display-mode: "form" }
"""PandasQuest - motor de corrección.

Se usa igual en el navegador (Pyodide), en los notebooks de Colab y en los tests.
Compara el `resultado` del alumno contra el de la solución oficial con tolerancia
a diferencias irrelevantes (tipos int/float, nombres de índice, dtype string, etc.).
"""
import math

import numpy as np
import pandas as pd


def _norm_s(s):
    dt = s.dtype
    if isinstance(dt, pd.CategoricalDtype) or pd.api.types.is_string_dtype(dt):
        s = s.astype(object)
    elif pd.api.types.is_extension_array_dtype(dt):
        if pd.api.types.is_numeric_dtype(dt) and not pd.api.types.is_bool_dtype(dt):
            s = s.astype("float64")
        else:
            s = s.astype(object)
    if s.dtype == object:
        s = s.where(s.notna(), None)
    return s


def _norm_index(idx):
    if isinstance(idx, pd.MultiIndex):
        return idx
    if isinstance(idx, pd.CategoricalIndex) or pd.api.types.is_string_dtype(idx.dtype):
        return pd.Index(list(idx), dtype=object)
    return idx


def _norm(o):
    if isinstance(o, pd.Series):
        o = _norm_s(o.copy())
    else:
        cols = list(o.columns)
        o = pd.DataFrame({i: _norm_s(o.iloc[:, i]) for i in range(o.shape[1])}, index=o.index)
        o.columns = _norm_index(pd.Index(cols))
    o.index = _norm_index(o.index)
    return o


def _sort(o, ignore_index):
    if ignore_index:
        o = o.reset_index(drop=True)
        try:
            if isinstance(o, pd.Series):
                o = o.sort_values(kind="mergesort")
            else:
                o = o.sort_values(by=list(o.columns), kind="mergesort")
        except TypeError:
            key = o.astype(str) if isinstance(o, pd.Series) else o.astype(str).agg("|".join, axis=1)
            o = o.iloc[np.argsort(key.to_numpy(), kind="mergesort")]
        return o.reset_index(drop=True)
    try:
        return o.sort_index()
    except TypeError:
        return o


def _tipo(x):
    nombres = {pd.DataFrame: "un DataFrame", pd.Series: "una Series"}
    return nombres.get(type(x), f"un {type(x).__name__}")


def _scalar_eq(u, e):
    if isinstance(e, (np.generic,)):
        e = e.item()
    if isinstance(u, (np.generic,)):
        u = u.item()
    if isinstance(e, float) or isinstance(u, float):
        try:
            u, e = float(u), float(e)
        except (TypeError, ValueError):
            return False
        if math.isnan(e):
            return math.isnan(u)
        return math.isclose(u, e, rel_tol=1e-6, abs_tol=1e-6)
    if isinstance(e, (list, tuple)) and isinstance(u, (list, tuple, np.ndarray, pd.Index)):
        u = list(u)
        return len(u) == len(e) and all(_scalar_eq(a, b) for a, b in zip(u, e))
    try:
        return bool(u == e)
    except Exception:
        return False


def _array_compare(u, e):
    if isinstance(u, (pd.DataFrame, pd.Series)):
        u = u.to_numpy()
    try:
        ua = np.asarray(u)
    except Exception:
        return False, f"Se esperaba un array de NumPy y tu resultado es {_tipo(u)}."
    if not isinstance(u, np.ndarray) and not isinstance(u, (list, tuple)):
        return False, f"Se esperaba un array de NumPy y tu resultado es {_tipo(u)}."
    if ua.shape != e.shape:
        return False, f"Forma distinta: tu array tiene shape {ua.shape} y se esperaba {e.shape}."
    num = np.issubdtype(e.dtype, np.number) or e.dtype == bool
    try:
        if num and (np.issubdtype(ua.dtype, np.number) or ua.dtype == bool):
            ok = np.allclose(ua.astype(float), e.astype(float), rtol=1e-6, atol=1e-6, equal_nan=True)
        else:
            ok = ua.tolist() == e.tolist()
    except Exception:
        ok = False
    return (True, "") if ok else (False, "La forma es correcta pero los valores no coinciden. Revisa el cálculo.")


def pq_compare(u, e, ordered=True, ignore_index=False):
    """Devuelve (ok, mensaje)."""
    if isinstance(e, np.ndarray):
        return _array_compare(u, e)
    if isinstance(e, (pd.DataFrame, pd.Series)):
        if not isinstance(u, type(e)):
            extra = ""
            if isinstance(e, pd.Series) and isinstance(u, pd.DataFrame):
                extra = " Pista: con una sola columna usa df['col'] (corchetes simples)."
            if isinstance(e, pd.DataFrame) and isinstance(u, pd.Series):
                extra = " Pista: usa doble corchete df[['col']] o .reset_index() / .to_frame()."
            return False, f"Tu resultado es {_tipo(u)}, pero se esperaba {_tipo(e)}.{extra}"
        if isinstance(e, pd.DataFrame):
            uc, ec = [str(c) for c in u.columns], [str(c) for c in e.columns]
            if uc != ec:
                if sorted(uc) == sorted(ec):
                    return False, f"Tienes las columnas correctas pero en otro orden. Se esperaba: {ec}"
                faltan = [c for c in ec if c not in uc]
                sobran = [c for c in uc if c not in ec]
                msg = "Las columnas no coinciden."
                if faltan:
                    msg += f" Faltan: {faltan}."
                if sobran:
                    msg += f" Sobran: {sobran}."
                return False, msg
        if u.shape != e.shape:
            return False, f"Forma distinta: tu resultado tiene {u.shape} y se esperaba {e.shape} (filas, columnas)."
        u2, e2 = _norm(u), _norm(e)
        if not ordered:
            u2, e2 = _sort(u2, ignore_index), _sort(e2, ignore_index)
        elif ignore_index:
            u2, e2 = u2.reset_index(drop=True), e2.reset_index(drop=True)
        kw = dict(check_dtype=False, check_names=False, check_index_type=False,
                  check_exact=False, rtol=1e-6, atol=1e-6)
        try:
            if isinstance(e2, pd.DataFrame):
                pd.testing.assert_frame_equal(u2, e2, check_column_type=False, check_freq=False, **kw)
            else:
                pd.testing.assert_series_equal(u2, e2, check_freq=False, **kw)
        except AssertionError as err:
            txt = str(err)
            if "index" in txt.lower() and "values are different" in txt.lower() and "iloc" not in txt:
                return False, "Los valores parecen correctos pero el índice no coincide. ¿Hace falta ordenar o usar reset_index()?"
            if ordered and not ignore_index:
                try:
                    pd.testing.assert_frame_equal(
                        u2.to_frame() if isinstance(u2, pd.Series) else u2,
                        e2.to_frame() if isinstance(e2, pd.Series) else e2,
                        check_like=True, **kw)
                except AssertionError:
                    pass
                else:
                    return False, "Mismos datos, pero en otro orden. Revisa cómo ordenas las filas."
            return False, "Los valores no coinciden con lo esperado. Revisa la lógica y vuelve a intentarlo."
        return True, ""
    if isinstance(u, (pd.DataFrame, pd.Series)):
        return False, f"Tu resultado es {_tipo(u)}, pero se esperaba un valor simple ({type(e).__name__}). ¿Te falta .iloc[0], .sum() o similar?"
    if _scalar_eq(u, e):
        return True, ""
    return False, f"Tu resultado es {u!r}, pero no es el valor esperado."


# ---------------------------------------------------------------- gráficos
def _plt():
    import matplotlib.pyplot as plt
    return plt


def _fig():
    plt = _plt()
    return plt.gcf() if plt.get_fignums() else None


def _ax(i=0):
    f = _fig()
    if f is None or len(f.axes) <= i:
        raise LookupError("no hay ningún gráfico dibujado")
    return f.axes[i]


def _legend(ax=None):
    ax = ax or _ax()
    lg = ax.get_legend()
    return [t.get_text() for t in lg.get_texts()] if lg else []


def _heights(ax=None):
    return [round(float(p.get_height()), 6) for p in (ax or _ax()).patches]


def _widths(ax=None):
    return [round(float(p.get_width()), 6) for p in (ax or _ax()).patches]


def _xticks(ax=None):
    return [t.get_text() for t in (ax or _ax()).get_xticklabels()]


def _hex(color):
    import matplotlib.colors as mcolors
    return mcolors.to_hex(color)


_HELPERS = {"_fig": _fig, "_ax": _ax, "_legend": _legend, "_heights": _heights,
            "_widths": _widths, "_xticks": _xticks, "_hex": _hex}


def pq_prepare():
    """Deja matplotlib limpio antes de ejecutar código (si está cargado)."""
    import sys
    if "matplotlib.pyplot" in sys.modules:
        plt = sys.modules["matplotlib.pyplot"]
        plt.close("all")
        plt.show = lambda *a, **k: None


def pq_check(ns, get_expected, opts):
    """Evalúa el ejercicio.

    ns = namespace del alumno. get_expected = función que devuelve el namespace de la
    solución (solo se ejecuta si hace falta, para no pisar los gráficos del alumno).
    """
    custom = opts.get("custom")
    if custom:
        if opts.get("plot") and _fig() is None:
            return False, "Todavía no hay ningún gráfico. ¡Dibuja algo! 🎨"
        env = dict(_HELPERS)
        env.update(ns)
        try:
            ok = bool(eval(custom, env))
        except Exception as err:
            return False, f"{opts.get('custom_msg', 'Todavía no es correcto.')} ({type(err).__name__}: {err})"
        return ok, "" if ok else opts.get("custom_msg", "Todavía no es correcto. ¡Sigue intentándolo!")
    if "resultado" not in ns:
        return False, "Guarda tu respuesta en la variable `resultado`."
    exp_ns = get_expected() if callable(get_expected) else get_expected
    return pq_compare(ns["resultado"], exp_ns["resultado"],
                      ordered=opts.get("ordered", True),
                      ignore_index=opts.get("ignore_index", False))


_ERR_TIPS = [
    ("truth value of a Series is ambiguous", "Para combinar condiciones usa & (y), | (o) y pon cada condición entre paréntesis."),
    ("KeyError", "Revisa que el nombre de la columna esté bien escrito (mayúsculas, tildes, espacios)."),
    ("NameError", "Estás usando una variable que no existe. ¿Un error de tipeo?"),
    ("SyntaxError", "Hay un error de sintaxis: revisa paréntesis, comillas y que la línea `resultado = ...` esté completa."),
    ("AttributeError", "Ese método o atributo no existe para este objeto. Revisa el nombre."),
    ("TypeError", "Algún tipo de dato no encaja (¿texto mezclado con números?)."),
    ("IndentationError", "Revisa la sangría (espacios al inicio de la línea)."),
    ("MergeError", "Revisa las columnas que usas para unir (on / left_on / right_on)."),
]


def pq_error_tip(msg):
    for key, tip in _ERR_TIPS:
        if key in msg:
            return tip
    return ""


_SETUP = 'import pandas as pd\nimport numpy as np\n\nclientes = pd.DataFrame({\n    "cliente_id": range(1, 13),\n    "nombre": ["Ana", "Bruno", "Carla", "Diego", "Elena", "Facundo",\n               "Gabriela", "Hugo", "Inés", "Julián", "Karina", "Lucas"],\n    "ciudad": ["Madrid", "Buenos Aires", "CDMX", "Bogotá", "Madrid", "Lima",\n               "Buenos Aires", "CDMX", "Santiago", "Bogotá", "Madrid", "Lima"],\n    "pais": ["España", "Argentina", "México", "Colombia", "España", "Perú",\n             "Argentina", "México", "Chile", "Colombia", "España", "Perú"],\n    "edad": [34, 28, 45, 23, 51, 39, 30, 27, 42, 36, 25, 48],\n    "segmento": ["Premium", "Básico", "Premium", "Básico", "Premium", "Estándar",\n                 "Estándar", "Básico", "Premium", "Estándar", "Básico", "Estándar"],\n    "email": ["ana@mail.com", None, "carla@mail.com", "diego@mail.com", None, "facu@mail.com",\n              "gabi@mail.com", None, "ines@mail.com", "julian@mail.com", "kari@mail.com", "lucas@mail.com"],\n    "fecha_registro": pd.to_datetime([\n        "2023-01-15", "2023-02-03", "2023-02-20", "2023-03-11", "2023-04-02", "2023-05-19",\n        "2023-06-07", "2023-07-23", "2023-08-30", "2023-09-14", "2023-10-05", "2023-11-28"]),\n})\n\nproductos = pd.DataFrame({\n    "producto_id": range(1, 11),\n    "producto": ["Laptop", "Mouse", "Teclado", "Monitor", "Auriculares",\n                 "Webcam", "Silla gamer", "Escritorio", "Tablet", "Micrófono"],\n    "categoria": ["Computación", "Accesorios", "Accesorios", "Computación", "Audio",\n                  "Accesorios", "Muebles", "Muebles", "Computación", "Audio"],\n    "precio": [1200.0, 25.5, 45.0, 300.0, 80.0, 60.0, 250.0, 180.0, 450.0, 95.0],\n    "stock": [5, 150, 80, 20, 60, 0, 12, 7, 15, 0],\n})\n\n_estados = ["entregado", "enviado", "entregado", "cancelado", "entregado", "pendiente"]\npedidos = pd.DataFrame({\n    "pedido_id": range(1001, 1031),\n    "cliente_id": [(i * 7) % 11 + 1 for i in range(29)] + [99],\n    "producto_id": [(i * 3) % 10 + 1 for i in range(30)],\n    "cantidad": [(i % 4) + 1 for i in range(30)],\n    "fecha": pd.date_range("2024-01-03", periods=30, freq="5D"),\n    "estado": [_estados[i % 6] for i in range(30)],\n})\n\nempleados = pd.DataFrame({\n    "emp_id": range(1, 11),\n    "nombre": ["Marta", "Pablo", "Sofía", "Tomás", "Valeria",\n               "Andrés", "Lucía", "Martín", "Paula", "Ramiro"],\n    "depto": ["Dirección", "Ventas", "Ventas", "Ventas", "IT",\n              "IT", "IT", "Marketing", "Marketing", "Ventas"],\n    "salario": [9000, 4200, 3900, 4000, 5200, 4800, 6100, 3700, 4100, 3500],\n    "jefe_id": pd.array([None, 1, 2, 2, 1, 5, 5, 1, 8, 2], dtype="Int64"),\n    "fecha_ingreso": pd.to_datetime([\n        "2015-03-01", "2018-06-15", "2019-01-10", "2020-09-01", "2017-11-20",\n        "2021-02-01", "2016-08-08", "2022-04-18", "2019-07-07", "2023-01-09"]),\n})\n\n_i = np.arange(90)\n_f = pd.date_range("2024-01-01", periods=90, freq="D")\n_v = 1000 + 8 * _i + 250 * (_f.dayofweek >= 5) + ((_i * 37) % 11) * 20\n_v = np.where(np.isin(_i, [20, 55, 77]), _v + 1500, _v)\nventas_diarias = pd.DataFrame({\n    "fecha": _f,\n    "ventas": _v.astype(float),\n    "visitas": (300 + (_i * 13) % 97 + 2 * _i).astype(int),\n})\n\nsucio = pd.DataFrame({\n    "id": [1, 2, 2, 3, 4, 5, 6, 7],\n    "nombre": ["  ana ", "BRUNO", "BRUNO", "carla", None, "Diego  ", "elena", "facundo"],\n    "edad": ["34", "28", "28", "n/d", "23", "51", None, "39"],\n    "ciudad": ["madrid", "Buenos Aires", "Buenos Aires", " santiago", "bogotá", "Madrid ", "madrid", "LIMA"],\n    "monto": ["$1,200.50", "$300", "$300", "$45.00", None, "$80.5", "$99", "$1,000"],\n})\ndel _i, _f, _v, _estados\n'
_SOL_B64 = 'eyJwZDAxIjogeyJzb2x1dGlvbiI6ICJyZXN1bHRhZG8gPSBwZC5TZXJpZXMoWzEwLCAyMCwgMzBdLCBpbmRleD1bJ2EnLCAnYicsICdjJ10pIiwgImNoZWNrIjoge30sICJ4cCI6IDEwfSwgInBkMDIiOiB7InNvbHV0aW9uIjogInZlbnRhcyA9IHsnTHVuJzogMTIwLCAnTWFyJzogOTUsICdNacOpJzogMTQzfVxucmVzdWx0YWRvID0gcGQuU2VyaWVzKHZlbnRhcykiLCAiY2hlY2siOiB7fSwgInhwIjogMTB9LCAicGQwMyI6IHsic29sdXRpb24iOiAidmVudGFzID0gcGQuU2VyaWVzKHsnTHVuJzogMTIwLCAnTWFyJzogOTUsICdNacOpJzogMTQzLCAnSnVlJzogODgsICdWaWUnOiAxNzB9KVxucmVzdWx0YWRvID0gW3ZlbnRhcy5sb2NbJ01pw6knXSwgdmVudGFzLmlsb2NbMF1dIiwgImNoZWNrIjoge30sICJ4cCI6IDIwfSwgInBkMDQiOiB7InNvbHV0aW9uIjogInByZWNpb3MgPSBwZC5TZXJpZXMoWzEwMCwgMjUwLCA4MCwgNDBdLCBpbmRleD1bJ01vdXNlJywgJ1RlY2xhZG8nLCAnV2ViY2FtJywgJ0NhYmxlJ10pXG5yZXN1bHRhZG8gPSBwcmVjaW9zICogMS4yMSIsICJjaGVjayI6IHt9LCAieHAiOiAxMH0sICJwZDA1IjogeyJzb2x1dGlvbiI6ICJwcmVjaW9zID0gcGQuU2VyaWVzKFsxMDAsIDI1MCwgODAsIDQwXSwgaW5kZXg9WydNb3VzZScsICdUZWNsYWRvJywgJ1dlYmNhbScsICdDYWJsZSddKVxucmVzdWx0YWRvID0gcHJlY2lvc1twcmVjaW9zID4gOTBdIiwgImNoZWNrIjoge30sICJ4cCI6IDIwfSwgInBkMDYiOiB7InNvbHV0aW9uIjogIm5vdGFzID0gcGQuU2VyaWVzKFs3LCA5LCA2LCA4LCAxMCwgNV0sIGluZGV4PVsnQW5hJywgJ0JydW5vJywgJ0NhcmxhJywgJ0RpZWdvJywgJ0VsZW5hJywgJ0ZhY3UnXSlcbnJlc3VsdGFkbyA9IG5vdGFzLm1lYW4oKSIsICJjaGVjayI6IHt9LCAieHAiOiAxMH0sICJwZDA3IjogeyJzb2x1dGlvbiI6ICJub3RhcyA9IHBkLlNlcmllcyhbNywgOSwgNiwgOCwgMTAsIDVdLCBpbmRleD1bJ0FuYScsICdCcnVubycsICdDYXJsYScsICdEaWVnbycsICdFbGVuYScsICdGYWN1J10pXG5yZXN1bHRhZG8gPSBub3Rhcy5zb3J0X3ZhbHVlcyhhc2NlbmRpbmc9RmFsc2UpIiwgImNoZWNrIjoge30sICJ4cCI6IDEwfSwgInBkMDgiOiB7InNvbHV0aW9uIjogInRlbXBlcmF0dXJhcyA9IHBkLlNlcmllcyhbMjEuNSwgTm9uZSwgMTkuOCwgMjUuMSwgTm9uZSwgMjIuNCwgTm9uZV0pXG5yZXN1bHRhZG8gPSB0ZW1wZXJhdHVyYXMuaXNuYSgpLnN1bSgpIiwgImNoZWNrIjoge30sICJ4cCI6IDIwfSwgInBkMDkiOiB7InNvbHV0aW9uIjogInRhbGxlcyA9IHBkLlNlcmllcyhbJ1MnLCAnTScsICdMJywgJ00nLCAnUyddKVxucmVzdWx0YWRvID0gdGFsbGVzLm1hcCh7J1MnOiAnQ2hpY28nLCAnTSc6ICdNZWRpYW5vJywgJ0wnOiAnR3JhbmRlJ30pIiwgImNoZWNrIjoge30sICJ4cCI6IDIwfSwgInBkMTAiOiB7InNvbHV0aW9uIjogInRlbXBlcmF0dXJhcyA9IHBkLlNlcmllcyhbMTguNSwgMjIuMCwgMjYuMywgMjAuMCwgMjUuMCwgMzEuMl0pXG5cbmRlZiBjbGFzaWZpY2FyKHQpOlxuICAgIGlmIHQgPCAyMDpcbiAgICAgICAgcmV0dXJuICdmcsOtbydcbiAgICBpZiB0IDw9IDI1OlxuICAgICAgICByZXR1cm4gJ3RlbXBsYWRvJ1xuICAgIHJldHVybiAnY2Fsb3InXG5cbnJlc3VsdGFkbyA9IHRlbXBlcmF0dXJhcy5hcHBseShjbGFzaWZpY2FyKSIsICJjaGVjayI6IHt9LCAieHAiOiAzNX0sICJwZDExIjogeyJzb2x1dGlvbiI6ICJub21icmVzID0gcGQuU2VyaWVzKFsnICBhbmEgJywgJ0JSVU5PJywgJ2NhcmxhICAnLCAnZElFR08nXSlcbnJlc3VsdGFkbyA9IG5vbWJyZXMuc3RyLnN0cmlwKCkuc3RyLnRpdGxlKCkiLCAiY2hlY2siOiB7fSwgInhwIjogMjB9LCAicGQxMiI6IHsic29sdXRpb24iOiAicmVzdWx0YWRvID0gcGQuRGF0YUZyYW1lKHsnanVnYWRvcic6IFsnTWVzc2knLCAnUm9uYWxkbycsICdNYmFwcMOpJ10sICdnb2xlcyc6IFszMCwgMjUsIDI4XX0pIiwgImNoZWNrIjoge30sICJ4cCI6IDEwfSwgInBkMTMiOiB7InNvbHV0aW9uIjogInJlc3VsdGFkbyA9IGNsaWVudGVzLnNoYXBlIiwgImNoZWNrIjoge30sICJ4cCI6IDEwfSwgInBkMTQiOiB7InNvbHV0aW9uIjogInJlc3VsdGFkbyA9IHByb2R1Y3Rvcy5oZWFkKDMpIiwgImNoZWNrIjoge30sICJ4cCI6IDEwfSwgInBkMTUiOiB7InNvbHV0aW9uIjogInJlc3VsdGFkbyA9IGxpc3QocGVkaWRvcy5jb2x1bW5zKSIsICJjaGVjayI6IHt9LCAieHAiOiAxMH0sICJwZDE2IjogeyJzb2x1dGlvbiI6ICJyZXN1bHRhZG8gPSBwcm9kdWN0b3MuZGVzY3JpYmUoKSIsICJjaGVjayI6IHt9LCAieHAiOiAxMH0sICJwZDE3IjogeyJzb2x1dGlvbiI6ICJyZXN1bHRhZG8gPSBjbGllbnRlc1sncGFpcyddLnZhbHVlX2NvdW50cygpIiwgImNoZWNrIjogeyJvcmRlcmVkIjogZmFsc2V9LCAieHAiOiAxMH0sICJzdDAxIjogeyJzb2x1dGlvbiI6ICJzdWVsZG9zID0gcGQuU2VyaWVzKFszMDAwMCwgMzUwMDAsIDQwMDAwLCA0NTAwMCwgNTAwMDAsIDU1MDAwLCA1MDAwMDBdKVxucmVzdWx0YWRvID0gW3N1ZWxkb3MubWVhbigpLCBzdWVsZG9zLm1lZGlhbigpXSIsICJjaGVjayI6IHt9LCAieHAiOiAxMH0sICJzdDAyIjogeyJzb2x1dGlvbiI6ICJtb250b3MgPSBwZC5TZXJpZXMoWzEwMDAwLCAxMjAwMCwgMTUwMDAsIDE1MDAwLCAxMjAwMCwgOTUwMCwgMTIwMDAsIDE1MDAwXSlcbnJlc3VsdGFkbyA9IG1vbnRvcy5tb2RlKCkudG9saXN0KCkiLCAiY2hlY2siOiB7fSwgInhwIjogMTB9LCAic3QwMyI6IHsic29sdXRpb24iOiAic3VlbGRvcyA9IHBkLlNlcmllcyhbMzAwMDAsIDM1MDAwLCA0MDAwMCwgNDUwMDAsIDUwMDAwLCA1NTAwMCwgNTAwMDAwXSlcbnJlc3VsdGFkbyA9IFtzdWVsZG9zLnF1YW50aWxlKDAuMjUpLCBzdWVsZG9zLnF1YW50aWxlKDAuNSksIHN1ZWxkb3MucXVhbnRpbGUoMC43NSldIiwgImNoZWNrIjoge30sICJ4cCI6IDIwfSwgInN0MDQiOiB7InNvbHV0aW9uIjogInN1ZWxkb3MgPSBwZC5TZXJpZXMoWzMwMDAwLCAzNTAwMCwgNDAwMDAsIDQ1MDAwLCA1MDAwMCwgNTUwMDAsIDUwMDAwMF0pXG5xMSA9IHN1ZWxkb3MucXVhbnRpbGUoMC4yNSlcbnEzID0gc3VlbGRvcy5xdWFudGlsZSgwLjc1KVxuaXFyID0gcTMgLSBxMVxucmVzdWx0YWRvID0gc3VlbGRvc1soc3VlbGRvcyA8IHExIC0gMS41ICogaXFyKSB8IChzdWVsZG9zID4gcTMgKyAxLjUgKiBpcXIpXSIsICJjaGVjayI6IHt9LCAieHAiOiAzNX0sICJzdDA1IjogeyJzb2x1dGlvbiI6ICJlZGFkZXMgPSBwZC5TZXJpZXMoWzEwLCAxMiwgMTIsIDE1LCAxOCwgMjAsIDIyXSlcbnJlc3VsdGFkbyA9IHJvdW5kKGVkYWRlcy5zdGQoKSAvIGVkYWRlcy5tZWFuKCksIDMpIiwgImNoZWNrIjoge30sICJ4cCI6IDIwfSwgInBkMTgiOiB7InNvbHV0aW9uIjogImltcG9ydCBpb1xuXG5jc3YgPSAnJydjaXVkYWQsaGFiaXRhbnRlcyxwYWlzXG5NYWRyaWQsMzMwMDAwMCxFc3Bhw7FhXG5MaW1hLDEwMDAwMDAwLFBlcsO6XG5Cb2dvdMOhLDc5MDAwMDAsQ29sb21iaWEnJydcblxucmVzdWx0YWRvID0gcGQucmVhZF9jc3YoaW8uU3RyaW5nSU8oY3N2KSkiLCAiY2hlY2siOiB7fSwgInhwIjogMjB9LCAicGQxOSI6IHsic29sdXRpb24iOiAicmVzdWx0YWRvID0gcHJvZHVjdG9zLnRvX2NzdihpbmRleD1GYWxzZSkiLCAiY2hlY2siOiB7fSwgInhwIjogMjB9fQ=='
_TABLAS = ['clientes', 'productos', 'pedidos', 'empleados', 'ventas_diarias', 'sucio']

import base64 as _b64, json as _json, re as _re, warnings as _w
_w.simplefilter("ignore")
_SOL = _json.loads(_b64.b64decode(_SOL_B64).decode("utf-8"))

def reiniciar_datos():
    """Vuelve a cargar todas las tablas limpias (cada ejercicio empieza de cero)."""
    globals().pop("resultado", None)
    exec(_SETUP, globals())
    pq_prepare()

def _codigo_celda():
    try:
        src = get_ipython().history_manager.input_hist_raw[-1]
    except Exception:
        return ""
    src = "\n".join(l for l in src.splitlines() if "comprobar(" not in l)
    return "\n".join(_re.sub(r"#.*$", "", l) for l in src.splitlines())

def comprobar(ej):
    info = _SOL[ej]
    def esperado():
        exp_ns = {}
        exec(_SETUP, exp_ns)
        exec(info["solution"], exp_ns)
        return exp_ns
    g = globals()
    g["_pq_code"] = _codigo_celda()
    ok, msg = pq_check(g, esperado, info["check"])
    if ok:
        from IPython.display import HTML, display
        display(HTML('<div style="padding:12px 16px;border-radius:12px;background:#0f9d6e;color:#fff;'
                     'font:600 15px system-ui">🎉 ¡Correcto! +' + str(info["xp"]) + ' XP · Sigue así 🐼</div>'))
    else:
        from IPython.display import HTML, display
        import html as _h
        display(HTML('<div style="padding:12px 16px;border-radius:12px;background:#b4235a;color:#fff;'
                     'font:500 15px system-ui">❌ Todavía no. ' + _h.escape(msg) + '</div>'))
    if "resultado" in g:
        return g["resultado"]

def ver_solucion(ej):
    print(_SOL[ej]["solution"])

reiniciar_datos()
print("✅ Listo." + (" Datos cargados: " + ", ".join(_TABLAS) if _TABLAS else " Ya puedes empezar."))


---
# 📌 Series: la pieza básica

## 1. Hola, pandas
`Básico` · **+10 XP**

**pandas** es la librería de Python para trabajar con datos en forma de tabla. Es la herramienta número uno de analistas, científicos e ingenieros de datos: con ella se leen archivos, se limpian datos, se calculan métricas y se preparan datos para modelos de Machine Learning.

Todo en pandas se construye con dos piezas:
- **Series**: una columna de datos con etiquetas.
- **DataFrame**: una tabla, que en el fondo es un conjunto de Series que comparten las mismas etiquetas.

Una Series tiene **valores** y un **índice** (las etiquetas de cada valor). Si no indicas el índice, pandas usa 0, 1, 2…

```python
import pandas as pd   # "pd" es el alias estándar que usa todo el mundo

temperaturas = pd.Series([22, 25, 19], index=["lun", "mar", "mié"])
print(temperaturas)
# lun    22
# mar    25
# mié    19
# dtype: int64
```
El `dtype` final indica el **tipo de dato** de los valores (int64 = números enteros).

> 💼 **En el trabajo:** cada columna que leas de un Excel, un CSV o una base de datos llega a pandas como una Series.

### 🎯 Tu misión
Crea una Series con los valores `10, 20, 30` y el índice `'a', 'b', 'c'`. Guárdala en la variable `resultado`.

<details><summary>💡 Pista</summary>

`pd.Series([10, 20, 30], index=['a', 'b', 'c'])`
</details>

<details><summary>🗄️ Equivalente en SQL</summary>

```sql
-- En SQL no hay Series: equivale a una sola columna
SELECT valor FROM tabla;
```
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

# pandas ya está importado como pd
# Escribe tu código aquí 👇
resultado = 

comprobar("pd01")

## 2. Series desde un diccionario
`Básico` · **+10 XP**

Muchas veces los datos ya vienen como **diccionario** de Python: pares `clave: valor`. Si le pasas un diccionario a `pd.Series`, las **claves pasan a ser el índice** y los **valores**, los datos.

```python
poblacion = pd.Series({"Madrid": 3.3, "Lima": 10.0, "Bogotá": 7.9})
# Madrid     3.3
# Lima      10.0
# Bogotá     7.9
# dtype: float64
```

Esto es muy útil porque el índice deja de ser un número sin significado y pasa a describir cada dato.

> 💡 El orden se respeta: pandas usa el mismo orden en que escribiste las claves.

### 🎯 Tu misión
Crea una Series a partir del diccionario `{'Lun': 120, 'Mar': 95, 'Mié': 143}`, que representa las ventas de tres días. Guárdala en `resultado`.

<details><summary>💡 Pista</summary>

`pd.Series(ventas)`
</details>

<details><summary>🗄️ Equivalente en SQL</summary>

```sql
SELECT dia, ventas FROM ventas_semana;
```
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

ventas = {'Lun': 120, 'Mar': 95, 'Mié': 143}
resultado = 

comprobar("pd02")

## 3. Etiqueta o posición
`Intermedio` · **+20 XP**

Hay dos formas de sacar un valor de una Series, y conviene distinguirlas bien desde el principio:

- `.loc[etiqueta]`: busca por la **etiqueta** del índice.
- `.iloc[posición]`: busca por la **posición** (0 es el primero y -1 el último, como en las listas de Python).

```python
s = pd.Series([22, 25, 19], index=["lun", "mar", "mié"])
s.loc["mar"]   # 25  -> por etiqueta
s.iloc[0]      # 22  -> primera posición
s.iloc[-1]     # 19  -> última posición
```

También puedes escribir `s["mar"]`, pero usar `loc` e `iloc` deja clarísimo qué quieres decir.

> ⚠️ **Error común:** usar `iloc` con una etiqueta (`s.iloc["mar"]`) da error, porque `iloc` solo acepta números de posición.

### 🎯 Tu misión
Con la Series `ventas`, guarda en `resultado` una **lista** con dos valores: las ventas del **Mié** (por etiqueta, con `loc`) y las ventas del **primer día** (por posición, con `iloc`).

<details><summary>💡 Pista</summary>

`[ventas.loc['Mié'], ventas.iloc[0]]`
</details>

<details><summary>🗄️ Equivalente en SQL</summary>

```sql
SELECT ventas FROM ventas_semana WHERE dia = 'Mié';
```
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

ventas = pd.Series({'Lun': 120, 'Mar': 95, 'Mié': 143, 'Jue': 88, 'Vie': 170})
resultado = [ , ]

comprobar("pd03")

---
# 📌 Operaciones con Series

## 4. Cálculos sin bucles
`Básico` · **+10 XP**

La gran ventaja de pandas son las **operaciones vectorizadas**: escribes la operación una vez y se aplica a **todos los valores a la vez**, sin bucles `for`. Además de más cómodo, es muchísimo más rápido.

```python
precios = pd.Series([100, 200], index=["A", "B"])
precios * 2        # A 200, B 400
precios + 10       # A 110, B 210
precios / 100      # A 1.0, B 2.0
```

El índice se conserva, así que cada resultado sigue sabiendo a qué producto pertenece.

> 💼 **En el trabajo:** aplicar impuestos, convertir monedas o pasar de gramos a kilos se hace con una sola línea como esta.

### 🎯 Tu misión
Los precios de la Series `precios` no incluyen el IVA. Calcula el precio **con un IVA del 21%** (multiplica por `1.21`) y guarda la Series en `resultado`.

<details><summary>💡 Pista</summary>

`precios * 1.21`
</details>

<details><summary>🗄️ Equivalente en SQL</summary>

```sql
SELECT producto, precio * 1.21 FROM productos;
```
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

precios = pd.Series([100, 250, 80, 40], index=['Mouse', 'Teclado', 'Webcam', 'Cable'])
resultado = 

comprobar("pd04")

## 5. Filtrar valores
`Intermedio` · **+20 XP**

Cuando comparas una Series con un valor (`precios > 90`), obtienes otra Series de **True/False**, llamada **máscara booleana**. Si la pones entre corchetes, pandas **se queda solo con los valores True**:

```python
edades = pd.Series([15, 32, 17, 45])
edades > 18            # False, True, False, True
edades[edades > 18]    # 32, 45
```

Operadores de comparación: `>`, `<`, `>=`, `<=`, `==` (igual) y `!=` (distinto).

> 💡 Piénsalo en dos pasos: primero **preguntas** (la máscara) y después **filtras** con esa respuesta.

### 🎯 Tu misión
De la Series `precios`, quédate solo con los productos que cuestan **más de 90**. Guarda el resultado en `resultado`.

<details><summary>💡 Pista</summary>

`precios[precios > 90]`
</details>

<details><summary>🗄️ Equivalente en SQL</summary>

```sql
SELECT * FROM productos WHERE precio > 90;
```
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

precios = pd.Series([100, 250, 80, 40], index=['Mouse', 'Teclado', 'Webcam', 'Cable'])
resultado = 

comprobar("pd05")

## 6. Estadística exprés
`Básico` · **+10 XP**

Las Series traen métodos para resumir datos en un solo número:

| Método | Qué calcula |
|---|---|
| `.sum()` | suma |
| `.mean()` | media (promedio) |
| `.median()` | mediana (el valor del medio) |
| `.min()` / `.max()` | mínimo / máximo |
| `.std()` | desviación estándar (cuánto se dispersan los datos) |
| `.count()` | cantidad de valores no nulos |

```python
notas = pd.Series([6, 8, 10])
notas.mean()   # 8.0
notas.max()    # 10
```

> 💼 **En el trabajo:** "¿cuál es el ticket medio?", "¿cuál fue la venta máxima?"… son preguntas que se responden con estos métodos.

### 🎯 Tu misión
Calcula la **nota media** de la Series `notas` y guárdala en `resultado`.

<details><summary>💡 Pista</summary>

`notas.mean()`
</details>

<details><summary>🗄️ Equivalente en SQL</summary>

```sql
SELECT AVG(nota) FROM notas;
```
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

notas = pd.Series([7, 9, 6, 8, 10, 5], index=['Ana', 'Bruno', 'Carla', 'Diego', 'Elena', 'Facu'])
resultado = 

comprobar("pd06")

## 7. Ordenar un ranking
`Básico` · **+10 XP**

`sort_values()` ordena los **valores** de menor a mayor. Con `ascending=False` se ordenan de mayor a menor, que es lo que normalmente quieres para un ranking. Cada valor conserva su etiqueta.

```python
goles = pd.Series({"Ana": 3, "Luis": 7, "Eva": 5})
goles.sort_values(ascending=False)
# Luis    7
# Eva     5
# Ana     3
```

Existe también `sort_index()`, que ordena por las **etiquetas** (alfabéticamente, por ejemplo).

### 🎯 Tu misión
Ordena la Series `notas` de la **nota más alta a la más baja** y guarda el resultado.

<details><summary>💡 Pista</summary>

`notas.sort_values(ascending=False)`
</details>

<details><summary>🗄️ Equivalente en SQL</summary>

```sql
SELECT * FROM notas ORDER BY nota DESC;
```
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

notas = pd.Series([7, 9, 6, 8, 10, 5], index=['Ana', 'Bruno', 'Carla', 'Diego', 'Elena', 'Facu'])
resultado = 

comprobar("pd07")

## 8. Datos que faltan
`Intermedio` · **+20 XP**

En los datos reales casi siempre **faltan valores**. pandas los representa como `NaN` (*Not a Number*). Para detectarlos:

- `.isna()` devuelve True donde falta el dato.
- `.notna()` devuelve lo contrario.

Como True cuenta como 1 y False como 0, `.isna().sum()` te dice **cuántos datos faltan**:

```python
s = pd.Series([1, None, 3, None])
s.isna()         # False, True, False, True
s.isna().sum()   # 2
```

> ⚠️ **Error común:** comparar con `== None` o `== np.nan` **no funciona**. Usa siempre `isna()`.

> 💼 **En el trabajo:** lo primero que se hace con un dataset nuevo es contar los nulos de cada columna.

### 🎯 Tu misión
La Series `temperaturas` tiene lecturas faltantes. Cuenta **cuántos valores faltan** y guarda ese número en `resultado`.

<details><summary>💡 Pista</summary>

`temperaturas.isna().sum()`
</details>

<details><summary>🗄️ Equivalente en SQL</summary>

```sql
SELECT COUNT(*) FROM lecturas WHERE temperatura IS NULL;
```
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

temperaturas = pd.Series([21.5, None, 19.8, 25.1, None, 22.4, None])
resultado = 

comprobar("pd08")

## 9. Traducir valores con map
`Intermedio` · **+20 XP**

`.map(diccionario)` reemplaza cada valor por el que le corresponde en el diccionario. Es perfecto para **traducir códigos** a textos legibles, o al revés.

```python
estado = pd.Series(["A", "C", "A"])
estado.map({"A": "Activo", "C": "Cancelado"})
# Activo, Cancelado, Activo
```

> ⚠️ Si un valor no está en el diccionario, el resultado es `NaN`. Revisa siempre que el diccionario cubra todos los casos.

### 🎯 Tu misión
Traduce los talles de la Series `talles`: `S` → `Chico`, `M` → `Mediano`, `L` → `Grande`. Guarda la Series traducida en `resultado`.

<details><summary>💡 Pista</summary>

`talles.map({'S': 'Chico', 'M': 'Mediano', 'L': 'Grande'})`
</details>

<details><summary>🗄️ Equivalente en SQL</summary>

```sql
SELECT CASE talle WHEN 'S' THEN 'Chico' WHEN 'M' THEN 'Mediano' ELSE 'Grande' END FROM ropa;
```
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

talles = pd.Series(['S', 'M', 'L', 'M', 'S'])
resultado = 

comprobar("pd09")

## 10. Tu propia función con apply
`Avanzado` · **+35 XP**

Cuando la lógica es más compleja que una simple operación, puedes escribir **tu propia función** y aplicarla a cada valor con `.apply(funcion)`:

```python
def clasificar(nota):
    if nota >= 6:
        return "aprobado"
    return "desaprobado"

notas.apply(clasificar)
```

También puedes usar una función anónima en una línea, con `lambda`:

```python
notas.apply(lambda n: "aprobado" if n >= 6 else "desaprobado")
```

> 💡 `apply` es flexible pero más lento que las operaciones vectorizadas. Úsalo cuando no haya una alternativa directa.

### 🎯 Tu misión
Escribe una función que clasifique cada temperatura de `temperaturas`:
- **"frío"** si es menor que 20,
- **"templado"** si está entre 20 y 25 (ambos incluidos),
- **"calor"** si es mayor que 25.

Aplícala con `apply` y guarda la Series resultante en `resultado`.

<details><summary>💡 Pista</summary>

`if t < 20: return 'frío' / if t <= 25: return 'templado' / return 'calor'`
</details>

<details><summary>🗄️ Equivalente en SQL</summary>

```sql
SELECT CASE WHEN t < 20 THEN 'frío' WHEN t <= 25 THEN 'templado' ELSE 'calor' END FROM lecturas;
```
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

temperaturas = pd.Series([18.5, 22.0, 26.3, 20.0, 25.0, 31.2])

def clasificar(t):
    # completa aquí
    pass

resultado = temperaturas.apply(clasificar)

comprobar("pd10")

## 11. Limpiar texto con .str
`Intermedio` · **+20 XP**

Las Series de texto tienen el accesor **`.str`**, con los métodos de texto de Python pero **vectorizados** (se aplican a todos los valores a la vez):

```python
s.str.strip()       # quita los espacios del principio y del final
s.str.lower()       # minúsculas
s.str.upper()       # MAYÚSCULAS
s.str.title()       # Primera Letra En Mayúscula
s.str.len()         # largo de cada texto
s.str.contains("x") # True si contiene "x"
```

Se pueden **encadenar** uno detrás de otro: `s.str.strip().str.lower()`.

> 💼 **En el trabajo:** los nombres escritos a mano por personas llegan con espacios de más y mayúsculas mezcladas. Normalizarlos es tarea de todos los días.

### 🎯 Tu misión
Limpia la Series `nombres`: quita los espacios sobrantes y déjalos en formato Título (`Ana`, `Bruno`...). Guarda el resultado.

<details><summary>💡 Pista</summary>

`nombres.str.strip().str.title()`
</details>

<details><summary>🗄️ Equivalente en SQL</summary>

```sql
SELECT INITCAP(TRIM(nombre)) FROM personas;
```
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

nombres = pd.Series(['  ana ', 'BRUNO', 'carla  ', 'dIEGO'])
resultado = 

comprobar("pd11")

---
# 📌 Del Series al DataFrame

## 12. Tu primer DataFrame
`Básico` · **+10 XP**

Un **DataFrame** es una tabla: filas y columnas, como una hoja de Excel. Cada columna es una Series y todas comparten el mismo índice (las etiquetas de las filas).

La forma más habitual de crear uno a mano es con un **diccionario de listas**: cada clave es el nombre de una columna y cada lista, sus valores.

```python
df = pd.DataFrame({
    "fruta": ["pera", "uva", "kiwi"],
    "kilos": [3, 5, 2],
})
#   fruta  kilos
# 0  pera      3
# 1   uva      5
# 2  kiwi      2
```

> ⚠️ Todas las listas deben tener **el mismo largo**; si no, pandas da error.

### 🎯 Tu misión
Crea un DataFrame con la columna `jugador` = `Messi, Ronaldo, Mbappé` y la columna `goles` = `30, 25, 28`, en ese orden. Guárdalo en `resultado`.

<details><summary>💡 Pista</summary>

`{'jugador': ['Messi', 'Ronaldo', 'Mbappé'], 'goles': [30, 25, 28]}`
</details>

<details><summary>🗄️ Equivalente en SQL</summary>

```sql
CREATE TABLE jugadores (jugador TEXT, goles INT);
INSERT INTO jugadores VALUES ('Messi',30),('Ronaldo',25),('Mbappé',28);
```
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

resultado = pd.DataFrame({
    # 'columna': [valores],
})

comprobar("pd12")

## 13. Conoce tus tablas
`Básico` · **+10 XP**

A partir de aquí tienes cargadas las tablas de una tienda online: `clientes`, `productos`, `pedidos`, `empleados`, `ventas_diarias` y `sucio` (datos desordenados para limpiar). Puedes verlas cuando quieras con el botón **📋 Tablas**.

Lo primero con cualquier tabla es saber **cuánto mide**:
- `df.shape` devuelve una tupla `(filas, columnas)`.
- `len(df)` devuelve solo la cantidad de filas.

```python
productos.shape   # (10, 5) -> 10 filas y 5 columnas
```

> ⚠️ `shape` es un **atributo**, no un método: se escribe sin paréntesis.

### 🎯 Tu misión
Guarda en `resultado` la forma (`shape`) de la tabla `clientes`.

<details><summary>💡 Pista</summary>

`clientes.shape (sin paréntesis)`
</details>

<details><summary>🗄️ Equivalente en SQL</summary>

```sql
SELECT COUNT(*) FROM clientes;  -- filas
```
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

resultado = 

comprobar("pd13")

## 14. Un vistazo rápido
`Básico` · **+10 XP**

Con tablas de miles o millones de filas no puedes mirarlo todo. Por eso existen:

- `df.head(n)`: las primeras *n* filas (5 si no indicas nada).
- `df.tail(n)`: las últimas *n* filas.
- `df.sample(n)`: *n* filas al azar.

```python
pedidos.head(3)
```

> 💼 **En el trabajo:** `head()` es lo primero que se ejecuta con cualquier archivo nuevo, para ver qué columnas trae y cómo lucen los datos.

### 🎯 Tu misión
Guarda en `resultado` las **3 primeras filas** de `productos`.

<details><summary>💡 Pista</summary>

`productos.head(3)`
</details>

<details><summary>🗄️ Equivalente en SQL</summary>

```sql
SELECT * FROM productos LIMIT 3;
```
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

resultado = 

comprobar("pd14")

## 15. Inventario de columnas
`Básico` · **+10 XP**

Dos atributos que usarás a diario:

- `df.columns`: los nombres de las columnas. Con `list(df.columns)` se convierten en una lista normal.
- `df.dtypes`: el **tipo de dato** de cada columna:
  - `int64`: enteros; `float64`: decimales;
  - `object` o `str`: texto;
  - `datetime64`: fechas; `bool`: verdadero/falso.

```python
list(productos.columns)
# ['producto_id', 'producto', 'categoria', 'precio', 'stock']
```

> 💡 `df.info()` muestra todo junto: columnas, tipos y cantidad de valores no nulos.

### 🎯 Tu misión
Guarda en `resultado` una **lista** con los nombres de las columnas de `pedidos`.

<details><summary>💡 Pista</summary>

`list(pedidos.columns)`
</details>

<details><summary>🗄️ Equivalente en SQL</summary>

```sql
SELECT column_name FROM information_schema.columns WHERE table_name = 'pedidos';
```
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

resultado = 

comprobar("pd15")

## 16. Resumen estadístico
`Básico` · **+10 XP**

`df.describe()` calcula de golpe las estadísticas principales de **todas las columnas numéricas**:

- `count`: cuántos valores hay;
- `mean` y `std`: media y desviación estándar;
- `min` y `max`;
- `25%`, `50%`, `75%`: percentiles (el 50% es la mediana).

Es la forma más rápida de detectar cosas raras: un precio negativo, una edad de 300 años, un stock en cero…

> 💼 **En el trabajo:** `describe()` es parte del **EDA** (*Exploratory Data Analysis*), el análisis exploratorio que se hace antes de cualquier trabajo serio.

### 🎯 Tu misión
Guarda en `resultado` el resumen estadístico de `productos`.

<details><summary>💡 Pista</summary>

`productos.describe()`
</details>

<details><summary>🗄️ Equivalente en SQL</summary>

```sql
SELECT COUNT(precio), AVG(precio), MIN(precio), MAX(precio) FROM productos;
```
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

resultado = 

comprobar("pd16")

## 17. ¿Cuántos de cada uno?
`Básico` · **+10 XP**

`value_counts()` cuenta **cuántas veces aparece cada valor** de una columna, ordenado de mayor a menor. Es probablemente el método más usado en análisis exploratorio.

```python
pedidos["estado"].value_counts()
# entregado    15
# enviado       5
# ...
```

Variantes útiles:
- `value_counts(normalize=True)`: proporciones en lugar de conteos;
- `.nunique()`: cuántos valores **distintos** hay;
- `.unique()`: cuáles son.

### 🎯 Tu misión
Cuenta cuántos clientes hay de cada `pais` y guarda la Series en `resultado`.

<details><summary>💡 Pista</summary>

`clientes['pais'].value_counts()`
</details>

<details><summary>🗄️ Equivalente en SQL</summary>

```sql
SELECT pais, COUNT(*) FROM clientes GROUP BY pais ORDER BY 2 DESC;
```
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

resultado = 

comprobar("pd17")

---
# 📌 Estadística descriptiva

## 18. Media vs mediana
`Básico` · **+10 XP**

La **media** (promedio) suma todos los valores y los divide por la cantidad. La **mediana** es el valor del **medio** cuando los datos están ordenados.

Parecen lo mismo, pero no lo son. Mira este ejemplo clásico de sueldos:

```python
sueldos = pd.Series([30000, 35000, 40000, 45000, 50000, 55000, 500000])
sueldos.mean()      # 107857.14  -> inflada por un solo sueldo de 500.000
sueldos.median()    # 45000.0    -> la mitad gana menos y la otra mitad más
```

Un solo valor extremo (el sueldo de un CEO, por ejemplo) arrastra la media hacia arriba, pero casi no mueve la mediana.

> 💼 **En el trabajo:** con sueldos, precios de viviendas o montos de compras (datos con valores extremos) se informa la **mediana**. La media se usa cuando los datos son parejos y sin extremos.

### 🎯 Tu misión
Guarda en `resultado` una lista con la **media** y la **mediana** de `sueldos`.

<details><summary>💡 Pista</summary>

`[sueldos.mean(), sueldos.median()]`
</details>

<details><summary>🗄️ Equivalente en SQL</summary>

```sql
SELECT AVG(sueldo) FROM empleados;  -- la mediana no es estándar en SQL
-- PostgreSQL: SELECT PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY sueldo) FROM empleados;
```
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

sueldos = pd.Series([30000, 35000, 40000, 45000, 50000, 55000, 500000])
resultado = [ , ]

comprobar("st01")

## 19. La moda
`Básico` · **+10 XP**

La **moda** es el valor que **más se repite**. Es la única medida de tendencia central que también sirve para datos de **texto** (categorías).

```python
talles = pd.Series(["M", "L", "M", "S", "M", "L"])
talles.mode()        # 0    M   -> devuelve una Series
talles.mode()[0]     # 'M'
```

`mode()` devuelve una **Series** y no un solo valor porque puede haber **empate**: si dos valores aparecen la misma cantidad de veces, los datos son **bimodales** y pandas devuelve los dos.

```python
pd.Series([10, 12, 12, 15, 15]).mode()   # 12 y 15
```

> 💼 **En el trabajo:** el talle más vendido, el producto más comprado o la respuesta más común de una encuesta son modas.

### 🎯 Tu misión
Los `montos` son las cuotas que pagaron los socios de un gimnasio. Guarda en `resultado` la **lista** de modas (usa `.mode()` y conviértelo con `.tolist()`).

<details><summary>💡 Pista</summary>

`montos.mode().tolist()`
</details>

<details><summary>🗄️ Equivalente en SQL</summary>

```sql
SELECT monto, COUNT(*) AS veces FROM pagos
GROUP BY monto ORDER BY veces DESC;
```
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

montos = pd.Series([10000, 12000, 15000, 15000, 12000, 9500, 12000, 15000])
resultado = 

comprobar("st02")

## 20. Cuartiles
`Intermedio` · **+20 XP**

Los **cuartiles** dividen los datos ordenados en **cuatro partes iguales** (25% cada una):

| Cuartil | Significado | pandas |
|---|---|---|
| **Q1** | el 25% de los datos está por debajo | `s.quantile(0.25)` |
| **Q2** | el 50% está por debajo (**es la mediana**) | `s.quantile(0.50)` |
| **Q3** | el 75% está por debajo | `s.quantile(0.75)` |

```python
s = pd.Series([10, 12, 12, 15, 18, 20, 22])
s.quantile([0.25, 0.5, 0.75])    # 12.0, 15.0, 19.0
```

Con los sueldos del ejemplo, el 50% del medio de la gente gana entre Q1 y Q3: ese es el **rango típico**.

> 💡 pandas calcula los cuartiles **interpolando** entre valores. Por eso a veces da un número que no está en los datos, o difiere un poco de lo que harías a mano. La conclusión es la misma.

### 🎯 Tu misión
Guarda en `resultado` una lista con **Q1, Q2 y Q3** de `sueldos`.

<details><summary>💡 Pista</summary>

`[sueldos.quantile(0.25), sueldos.quantile(0.5), sueldos.quantile(0.75)]`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

sueldos = pd.Series([30000, 35000, 40000, 45000, 50000, 55000, 500000])
resultado = 

comprobar("st03")

## 21. Detectar outliers con IQR
`Avanzado` · **+35 XP**

Un **outlier** (valor atípico) es un dato muy alejado del resto. Puede ser un error de carga o un caso especial real, y siempre hay que revisarlo.

El **criterio de cuartiles** (el mismo que usan los diagramas de caja) dice:

1. **IQR** = Q3 − Q1 → el ancho del 50% central
2. **Límite inferior** = Q1 − 1.5 × IQR
3. **Límite superior** = Q3 + 1.5 × IQR
4. Todo lo que quede **afuera** de los límites es outlier.

```python
q1, q3 = s.quantile(0.25), s.quantile(0.75)
iqr = q3 - q1
outliers = s[(s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)]
```

Con los sueldos de clase: Q1 = 37.500, Q3 = 52.500, IQR = 15.000 → rango normal de 15.000 a 75.000. El sueldo de **500.000** queda afuera: es el outlier.

> ⚠️ Detectar un outlier no significa borrarlo. Primero entiende **por qué** está ahí.

### 🎯 Tu misión
Calcula los límites con el criterio IQR y guarda en `resultado` la **Series** con los outliers de `sueldos`.

<details><summary>💡 Pista</summary>

`sueldos[(sueldos < q1 - 1.5 * iqr) | (sueldos > q3 + 1.5 * iqr)]`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

sueldos = pd.Series([30000, 35000, 40000, 45000, 50000, 55000, 500000])
q1 = sueldos.quantile(0.25)
q3 = sueldos.quantile(0.75)
iqr = 
resultado = 

comprobar("st04")

## 22. Desviación estándar
`Intermedio` · **+20 XP**

La **desviación estándar** mide cuánto se alejan **típicamente** los datos de la media:

- **Baja**: los datos están agrupados cerca de la media (son predecibles).
- **Alta**: los datos están dispersos.

```python
pd.Series([5, 5, 5, 5]).std()       # 0.0   -> todos iguales
pd.Series([2, 8]).std()             # 4.24  -> muy dispersos
```

Para comparar la dispersión de datos con escalas distintas (sueldos en pesos contra edades en años) se usa el **coeficiente de variación**: `std / mean`. Sin unidades, se puede comparar.

> 💡 pandas calcula la desviación **muestral** (divide por n − 1). NumPy (`np.std`) usa por defecto la **poblacional** (divide por n). Por eso a veces dan distinto.

### 🎯 Tu misión
Guarda en `resultado` el **coeficiente de variación** (`std / mean`) de las `edades` de un grupo, **redondeado a 3 decimales** con `round(valor, 3)`.

<details><summary>💡 Pista</summary>

`round(edades.std() / edades.mean(), 3)`
</details>

<details><summary>🗄️ Equivalente en SQL</summary>

```sql
SELECT STDDEV(edad) / AVG(edad) FROM grupo;
```
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

edades = pd.Series([10, 12, 12, 15, 18, 20, 22])
resultado = 

comprobar("st05")

---
# 📌 Leer y guardar datos

## 23. Leer un CSV
`Intermedio` · **+20 XP**

El **CSV** (valores separados por comas) es el formato de datos más común del mundo. pandas lo lee con `pd.read_csv()`:

```python
df = pd.read_csv("ventas.csv")                 # archivo local
df = pd.read_csv("https://.../datos.csv")      # desde una URL
df = pd.read_csv("datos.csv", sep=";")         # separado por punto y coma
```

En **Google Colab** puedes subir un archivo desde el panel de la izquierda (📁) o leerlo directamente desde Google Drive.

Aquí vamos a leer un CSV que está guardado en un **texto**. Para que pandas lo trate como un archivo se usa `io.StringIO(texto)`.

> 💡 Otros lectores: `pd.read_excel()`, `pd.read_json()`, `pd.read_parquet()` y `pd.read_sql()` (este último lo verás en el módulo de DataFrames).

### 🎯 Tu misión
Lee el texto `csv` con `pd.read_csv` (envuelto en `io.StringIO`) y guarda el DataFrame en `resultado`.

<details><summary>💡 Pista</summary>

`pd.read_csv(io.StringIO(csv))`
</details>

<details><summary>🗄️ Equivalente en SQL</summary>

```sql
COPY ciudades FROM 'ciudades.csv' CSV HEADER;  -- PostgreSQL
```
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

import io

csv = '''ciudad,habitantes,pais
Madrid,3300000,España
Lima,10000000,Perú
Bogotá,7900000,Colombia'''

resultado = 

comprobar("pd18")

## 24. Exportar resultados
`Intermedio` · **+20 XP**

Cuando terminas un análisis hay que **compartirlo**. Los métodos `to_...` guardan un DataFrame:

```python
df.to_csv("resultado.csv", index=False)    # index=False: no guarda el índice
df.to_excel("reporte.xlsx", index=False)
df.to_json("datos.json")
```

Si no indicas el archivo, `to_csv()` **devuelve el texto** del CSV, lo que viene bien para verlo o enviarlo.

> ⚠️ **Error común:** olvidar `index=False` y acabar con una columna extra `Unnamed: 0` al volver a leer el archivo.

### 🎯 Tu misión
Convierte `productos` a texto CSV **sin el índice** y guarda ese texto en `resultado`.

<details><summary>💡 Pista</summary>

`productos.to_csv(index=False)`
</details>

<details><summary>🗄️ Equivalente en SQL</summary>

```sql
COPY productos TO 'productos.csv' CSV HEADER;  -- PostgreSQL
```
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

resultado = 

comprobar("pd19")

---
# 🏁 ¡Módulo completado!
Vuelve a [PandasQuest](https://rosalesluciano.github.io/pandas-quest/) para sumar tus XP y seguir con el siguiente módulo. 🚀